# RG3 독립 검증 자료 현재 상태 재감사

새 모델을 돌리는 실험이 아닌 데이터 출처/사용 이력/봉인 재검증. 기존591 + 개봉37 = 원본628 RH 키의 소진 여부를 현재 모든 CSV 메타데이터로 대조. 비라벨은 평가 정답 아님. identity 없는 CN7 명칭 파일은 RG3 독립 라벨로 가정하지 않음. 압축CSV 해시, 현재 로컬 Git refs 데이터 바이트, 봉인 프로토콜/예측 해시 재확인. 원격은 새 fetch하지 않았으며 외부 숨은 데이터의 존재를 부정하는 감사가 아님. 봉인은 사전 기록의 무결성 증거이지 모든 과거 비저장 조회가 없었다는 증거는 아님.


In [1]:
from pathlib import Path
import json,hashlib,zipfile,subprocess
import pandas as pd,numpy as np
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
keys=['EQUIP_CD','PART_NAME','TimeStamp']
old=pd.read_csv(root/'data/audit/rg3_rh_labeled_with_timestamp.csv',usecols=keys,parse_dates=['TimeStamp'])
used_test=pd.read_csv(t/'rg3_independent37_unblinded_results.csv',parse_dates=['TimeStamp'])
used=pd.concat([old,used_test[keys]]).drop_duplicates(keys)
inventory=[];remaining=[]
for path in sorted((root/'data').rglob('*.csv')):
 cols=pd.read_csv(path,nrows=0).columns.tolist()
 row=dict(path=str(path.relative_to(root)),sha256=hashlib.sha256(path.read_bytes()).hexdigest(),bytes=path.stat().st_size,has_quality_label='PassOrFail' in cols,has_identity=all(k in cols for k in keys))
 if not row['has_quality_label']:
  row['status']='No quality truth: not independent gas evaluation'
 elif row['has_identity']:
  x=pd.read_csv(path,usecols=keys,parse_dates=['TimeStamp']).drop_duplicates(keys)
  rh=x[x.PART_NAME.str.contains('RG3',na=False)&x.PART_NAME.str.contains('RH',na=False)]
  novel=rh.merge(used.assign(used=True),on=keys,how='left')
  novel=novel[novel.used.isna()]
  row.update(RG3_RH_unique_keys=len(rh),RG3_RH_keys_without_used_trace=len(novel),status='Identified RG3 RH labels already consumed' if len(rh) else 'Other product/side; not new RG3 RH labels')
  if len(novel):remaining.append(novel[keys].assign(source=str(path.relative_to(root))))
 else:
  row['status']='Identity absent: source/copy verification needed; cannot certify independent RG3 truth'
 inventory.append(row)
inv=pd.DataFrame(inventory)
for i,row in inv.iterrows():
 if row.has_quality_label and not row.has_identity:
  identified=inv[(inv.sha256==row.sha256)&(inv.has_identity)]
  copies=inv[(inv.sha256==row.sha256)&(inv.path!=row.path)]
  inv.loc[i,'identical_copy_paths']=';'.join(copies.path)
  if 'cn7' in row.path.lower():inv.loc[i,'status']='CN7-named identity-free source; not certified RG3 labels. Do not relabel as RG3 or assume human-verified truth.'
raw=pd.read_csv(root/'data/original/labeled_data.csv',usecols=keys,parse_dates=['TimeStamp'])
rh=raw[raw.PART_NAME.str.contains('RG3',na=False)&raw.PART_NAME.str.contains('RH',na=False)].drop_duplicates(keys)
assert len(rh)==628 and len(old)==591 and len(used_test)==37 and len(used)==628
assert not remaining
seal=pd.read_csv(t/'rg3_independent37_prediction_seal.csv').iloc[0]
predpath=t/'rg3_independent37_predictions_SEALED.csv'
protocol=root/'reports/rg3_independent37_locked_protocol.json'
assert hashlib.sha256(predpath.read_bytes()).hexdigest()==seal.prediction_sha256
assert hashlib.sha256(protocol.read_bytes()).hexdigest()==seal.protocol_sha256
result=used_test
assert len(result)==37 and result.actual_gas.sum()==2
checks=[]
for col in ['primary_prediction','reference_prediction','secondary_prediction']:
 tn,fp,fn,tp=confusion_matrix(result.actual_gas,result[col],labels=[0,1]).ravel()
 assert (tp,fp,fn,tn)==(0,12,2,23)
 checks.append(dict(model=col,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),sealed_prediction_hash_valid=True,sealed_protocol_hash_valid=True))
archives=[]
with zipfile.ZipFile(root/'data/1. 사출성형기 AI 데이터셋.zip') as z:
 for entry in z.namelist():
  if entry.endswith('.csv'):
   local=root/'data'/Path(entry).name
   same=local.exists() and hashlib.sha256(z.read(entry)).hexdigest()==hashlib.sha256(local.read_bytes()).hexdigest()
   archives.append(dict(entry=entry,identical_current_file=bool(same)))
assert len(archives)==4 and all(a['identical_current_file'] for a in archives)
refs=subprocess.check_output(['git','for-each-ref','--format=%(refname)','refs/heads','refs/remotes'],cwd=root,text=True).splitlines()
branches=[]
for ref in refs:
 paths=subprocess.check_output(['git','ls-tree','-r','--name-only',ref,'--','data'],cwd=root,text=True).splitlines()
 for path in paths:
  if path.endswith(('.csv','.zip')):
   blob=subprocess.check_output(['git','show',ref+':'+path],cwd=root)
   h=hashlib.sha256(blob).hexdigest()
   branches.append(dict(ref=ref,path=path,sha256=h,identical_current_data=bool((inv.sha256==h).any()) if path.endswith('.csv') else h==hashlib.sha256((root/'data/1. 사출성형기 AI 데이터셋.zip').read_bytes()).hexdigest()))
assert all(row['identical_current_data'] for row in branches)
inv.to_csv(t/'rg3_current_independent_source_inventory.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(checks).to_csv(t/'rg3_current_seal_revalidation.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(archives).to_csv(t/'rg3_current_archive_revalidation.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(branches,columns=['ref','path','sha256','identical_current_data']).to_csv(t/'rg3_current_local_branch_data_audit.csv',index=False,encoding='utf-8-sig')
summary=dict(scope='Workspace files and current local branch/ref snapshots; not freshly fetched remote or external hidden datasets',csv_sources=len(inv),RG3_RH_original_keys=628,previous_keys=591,consumed_extra_keys=37,unused_identifiable_RG3_RH_labels=0,blind_extra_gas=2,blind_extra_normals=35,prediction_hash_valid=True,protocol_hash_valid=True,archive_csv_copies=4,local_refs_checked=len(refs),local_ref_data_entries=len(branches),independent_improvement_confirmed=False,identity_free_CN7_source_not_certified_for_RG3=True)
(t/'rg3_current_independent_completion_audit.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
print(pd.DataFrame(checks).to_string(index=False))


{
  "scope": "Workspace files and current local branch/ref snapshots; not freshly fetched remote or external hidden datasets",
  "csv_sources": 22,
  "RG3_RH_original_keys": 628,
  "previous_keys": 591,
  "consumed_extra_keys": 37,
  "unused_identifiable_RG3_RH_labels": 0,
  "blind_extra_gas": 2,
  "blind_extra_normals": 35,
  "prediction_hash_valid": true,
  "protocol_hash_valid": true,
  "archive_csv_copies": 4,
  "local_refs_checked": 7,
  "local_ref_data_entries": 0,
  "independent_improvement_confirmed": false,
  "identity_free_CN7_source_not_certified_for_RG3": true
}
               model  TP  FP  FN  TN  sealed_prediction_hash_valid  sealed_protocol_hash_valid
  primary_prediction   0  12   2  23                          True                        True
reference_prediction   0  12   2  23                          True                        True
secondary_prediction   0  12   2  23                          True                        True
